In [ ]:
import os

import numpy as np
import pyarrow.compute as pc
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer

# Valores default do antigo parser agora como variáveis
tokenizer_filename = "NousResearch/Llama-2-7b-hf"
dataset_filename = "carolina-c4ai/corpus-carolina"
taxonomy = "leg"
out_dir = "processed"
dtype_str = "uint16"
num_proc = 10
total_batch = 32
chunk_length = 2048

# Resolução do Dtype correspondente no NumPy
if dtype_str == "uint16":
    dtype = np.uint16
elif dtype_str == "int16":
    dtype = np.int16
elif dtype_str == "uint32":
    dtype = np.uint32
elif dtype_str == "int32":
    dtype = np.int32
else:
    raise ValueError(f"Dtype não suportado: {dtype_str}")

tokenizer = AutoTokenizer.from_pretrained(tokenizer_filename)


def process_batch(example: list):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=chunk_length,
        stride=0,
        return_overflowing_tokens=True,
        return_length=False,
    )

    tokenized["num_tokens"] = [len(ids) for ids in tokenized["input_ids"]]

    return tokenized


dataset = load_dataset(dataset_filename, taxonomy=taxonomy)

split_dataset = dataset["corpus"].train_test_split(
    test_size=0.05, seed=42, shuffle=True
)
split_dataset["val"] = split_dataset.pop("test")
split_dataset = split_dataset.remove_columns("meta")

tokenized = split_dataset.map(
    process_batch,
    desc="Tokenizando os dados",
    num_proc=5,
    remove_columns=["text"],
    batched=True,
    batch_size=5,
    writer_batch_size=100,
)

tokenized.set_format(type="numpy", columns=["input_ids"], dtype=dtype)

Tokenizando os dados (num_proc=5):  14%|█▍        | 535/3782 [01:58<10:21,  5.23 examples/s]

In [ ]:
out_dir_taxonomy = os.path.join(out_dir, taxonomy)
os.makedirs(out_dir_taxonomy, exist_ok=True)

'processed/soc'

In [34]:
for split, dt in tokenized.items():
    len_arrow = dt.data.column("num_tokens")
    arr_len = pc.sum(len_arrow).as_py()

    filename = os.path.join(out_dir_taxonomy, f"{taxonomy}_{split}.bin")

    arr = np.memmap(filename=filename, dtype=dtype, mode="w+", shape=(arr_len,))

    if len(dataset["corpus"]) > total_batch:
        idx = 0

        for batch_idx in tqdm(
            range(total_batch), desc=f"Escrevendo em {filename}"
        ):
            batch = dt.shard(
                num_shards=total_batch, index=batch_idx, contiguous=True
            )

            ids = np.concatenate(batch["input_ids"], dtype=dtype)
            arr[idx : idx + len(ids)] = ids
            idx += len(ids)
    else:
        ids = np.concatenate(dt["input_ids"], dtype=dtype)
        arr[: len(ids)] = ids

    arr.flush()

Escrevendo em processed/soc/soc_val.bin: 100%|██████████| 32/32 [00:00<00:00, 1065.58it/s]


In [ ]:
import os

import numpy as np
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer

# ---------------------------------------------------------
# Configuração
# ---------------------------------------------------------
tokenizer_filename = "NousResearch/Llama-2-7b-hf"
dataset_filename = "carolina-c4ai/corpus-carolina"
out_dir = "processed"
dtype_str = "uint16"
chunk_length = 2048  # tamanho de cada chunk de tokens (ajuste ao seu treino)
val_ratio = 0.05
seed = 42

# Todas as taxonomias do corpus carolina.
# Processamos uma de cada vez, sequencialmente - nunca o corpus
# inteiro (15GB) precisa estar em memória ao mesmo tempo.
TAXONOMIES = ["wik", "dat", "leg", "jud", "uni", "soc", "pub"]

if dtype_str == "uint16":
    dtype = np.uint16
elif dtype_str == "int16":
    dtype = np.int16
elif dtype_str == "uint32":
    dtype = np.uint32
elif dtype_str == "int32":
    dtype = np.int32
else:
    raise ValueError(f"Dtype não suportado: {dtype_str}")

tokenizer = AutoTokenizer.from_pretrained(tokenizer_filename)
os.makedirs(out_dir, exist_ok=True)


def tokenize_and_chunk(text: str):
    """
    Tokeniza um único texto e devolve uma lista de chunks (listas de ids),
    usando return_overflowing_tokens para que o tokenizer Rust faça a
    divisão em janelas - nunca segurando a sequência inteira do
    documento como uma lista intermediária além do necessário.
    """
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=chunk_length,
        stride=0,
        return_overflowing_tokens=True,
    )
    return tokenized["input_ids"]  # lista de listas (um item por chunk)


def process_taxonomy(taxonomy: str):
    print(f"\n=== Processando taxonomia: {taxonomy} ===")

    # streaming=True: os dados são lidos sob demanda, shard por shard,
    # nunca materializando a taxonomia inteira em memória de uma vez.
    ds = load_dataset(
        dataset_filename,
        taxonomy=taxonomy,
        streaming=True,
    )["corpus"]

    # Sem shuffle global (você confirmou que não precisa).
    # Fazemos split treino/val determinístico via hash do índice,
    # já que IterableDataset não tem train_test_split nativo.
    def is_val(idx: int) -> bool:
        # divisão determinística e reprodutível, sem precisar
        # conhecer o tamanho total do dataset de antemão
        return (idx % 1000) < int(val_ratio * 1000)

    # Arquivos temporários por taxonomia (depois concatenamos, se quiser
    # um único .bin final por split - ver nota ao final do script)
    train_path = os.path.join(out_dir, f"{taxonomy}_train.bin")
    val_path = os.path.join(out_dir, f"{taxonomy}_val.bin")

    # Como streaming não permite saber o total de tokens de antemão,
    # escrevemos em modo "append via arquivo aberto em bytes", não via
    # np.memmap de tamanho fixo. Isso evita precisar de duas passadas.
    train_f = open(train_path, "wb")
    val_f = open(val_path, "wb")

    n_docs = 0
    n_chunks = 0

    try:
        for idx, example in enumerate(tqdm(ds, desc=f"Tokenizando {taxonomy}")):
            text = example["text"]
            if not text:
                continue

            chunks = tokenize_and_chunk(text)  # 1 documento por vez -
            # nunca acumula batch gigante

            target_f = val_f if is_val(idx) else train_f

            for chunk in chunks:
                arr = np.array(chunk, dtype=dtype)
                target_f.write(arr.tobytes())
                n_chunks += 1

            n_docs += 1

            # descarta explicitamente - importante em loop longo
            del chunks
    finally:
        train_f.close()
        val_f.close()

    print(
        f"{taxonomy}: {n_docs} documentos -> {n_chunks} chunks de {chunk_length} tokens"
    )


if __name__ == "__main__":
    for taxonomy in TAXONOMIES:
        process_taxonomy(taxonomy)

    print("\nConcluído. Arquivos gerados em:", out_dir)
    print("Um par (train.bin, val.bin) por taxonomia.")
    print(
        "Se quiser um único train.bin / val.bin unificado, concatene os "
        "arquivos binários por split (são todos uint16 puro, sem "
        "cabeçalho, então é só um `cat` / concatenação binária direta)."
    )

/home/koheiseko/Documents/projects/language-model-from-scratch/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Processando taxonomia: wik ===


Tokenizando wik: 353105it [12:57, 555.10it/s] 